# Export Badini OmniVoice for offline browser use\n\nFirst set **Runtime → Change runtime type → T4 GPU** in Colab. This notebook runs once to convert the fine-tuned checkpoint into ONNX files. It is not an inference server. After export, download the bundle ZIP, extract it in the iPad Files app, and import its files into Badini Voice.\n\nDefault checkpoint: `Dezheen89/omnivoice-badini`. Change `MODEL_ID` if you intend to export another compatible OmniVoice fine-tune.

In [ ]:
!rm -rf /content/omnivoice-src /content/badini-omnivoice-web
!git clone --depth 1 https://github.com/AnEntrypoint/omnivoice.git /content/omnivoice-src
!git clone --depth 1 https://github.com/1alind/badini-omnivoice-web.git /content/badini-omnivoice-web
%cd /content/omnivoice-src
!pip install -q -e . onnx onnxscript huggingface_hub safetensors s3prl


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

MODEL_ID = "Dezheen89/omnivoice-badini"
OUT = Path("/content/badini-model-bundle")
OUT.mkdir(exist_ok=True)
print("Checkpoint:", MODEL_ID)
print("GPU:", __import__('torch').cuda.is_available())


In [ ]:
!python /content/badini-omnivoice-web/tools/export_onnx.py --model_path "$MODEL_ID" --output_dir "$OUT" --device cuda --dtype fp16


In [ ]:
!python /content/badini-omnivoice-web/tools/validate_bundle.py "$OUT"\n

## Download the bundle

The ZIP may be over 1 GB. Keep enough free space on the iPad. After downloading, open the ZIP in Files to extract it. In Badini Voice, select all extracted files together. Do **not** upload the model weights to the GitHub repository.

In [ ]:
zip_path = "/content/badini-omnivoice-model.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p, arcname=p.name)
print("ZIP size (GB):", round(Path(zip_path).stat().st_size / 1e9, 2))
files.download(zip_path)